## Q3 — Malimg (CNN)


# Q3 — Malimg (CNN + bias analysis)

## Checklist
- Load dataset and inspect class balance
- Train baseline CNN
- Apply imbalance mitigation and compare
- Analyze what features CNN relies on
- Reflection: fragility, interpretability


### Load data
I use the Malimg dataset with explicit train/val/test splits.


In [11]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score

DATA_DIR = "../data/raw/malimg"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR   = os.path.join(DATA_DIR, "val")
TEST_DIR  = os.path.join(DATA_DIR, "test")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cpu')

### Transforms and datasets
I keep preprocessing minimal: resize, to tensor, and normalize.


In [12]:
IMG_SIZE = 128
BATCH_SIZE = 64

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

train_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_tf)
val_ds   = datasets.ImageFolder(VAL_DIR, transform=eval_tf)
test_ds  = datasets.ImageFolder(TEST_DIR, transform=eval_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

num_classes = len(train_ds.classes)
num_classes


25

### Class imbalance overview
Malimg is imbalanced; I inspect the class counts before training.


In [13]:
from collections import Counter

train_counts = Counter([y for _, y in train_ds.samples])
class_counts = np.array([train_counts[i] for i in range(num_classes)])
class_counts


array([  97,   92, 2359, 1272,  158,   84,  116,  160,  141,  129,  304,
        344,  170,  147,   98,  127,  108,  113,  126,   64,  102,  105,
        326,   77,  640])

### Simple CNN (from scratch)
I use a small CNN to keep the architecture interpretable and aligned with the exam rules.


In [14]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * (IMG_SIZE//8) * (IMG_SIZE//8), 256), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = SimpleCNN(num_classes).to(device)
model


SimpleCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=32768, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=256, out_features=25, bias=True)
  )
)

### Training utilities
I track accuracy on train/val to compare baseline vs imbalance-aware training.


In [15]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.set_grad_enabled(is_train):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            if is_train:
                optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            if is_train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * xb.size(0)
            preds = out.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += xb.size(0)
    return total_loss / total, correct / total


### Baseline training (no class weights)
This is the simple model used for comparison.


In [16]:
EPOCHS = 3  # increase if time allows
baseline = SimpleCNN(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(baseline.parameters(), lr=1e-3)

train_hist = []
val_hist = []
for epoch in range(EPOCHS):
    train_loss, train_acc = run_epoch(baseline, train_loader, criterion, optimizer)
    val_loss, val_acc = run_epoch(baseline, val_loader, criterion)
    train_hist.append((train_loss, train_acc))
    val_hist.append((val_loss, val_acc))
    print(f"Epoch {epoch+1}/{EPOCHS} | train_acc={train_acc:.3f} | val_acc={val_acc:.3f}")


Epoch 1/3 | train_acc=0.865 | val_acc=0.973
Epoch 2/3 | train_acc=0.966 | val_acc=0.970
Epoch 3/3 | train_acc=0.973 | val_acc=0.971


### Imbalance handling: class-weighted loss
I compare the baseline against a weighted loss that up-weights minority classes.


In [17]:
weights = (class_counts.sum() / (class_counts + 1e-8))
weights = torch.tensor(weights, dtype=torch.float32).to(device)

weighted = SimpleCNN(num_classes).to(device)
criterion_w = nn.CrossEntropyLoss(weight=weights)
optimizer_w = torch.optim.Adam(weighted.parameters(), lr=1e-3)

train_hist_w = []
val_hist_w = []
for epoch in range(EPOCHS):
    train_loss, train_acc = run_epoch(weighted, train_loader, criterion_w, optimizer_w)
    val_loss, val_acc = run_epoch(weighted, val_loader, criterion_w)
    train_hist_w.append((train_loss, train_acc))
    val_hist_w.append((val_loss, val_acc))
    print(f"Epoch {epoch+1}/{EPOCHS} | train_acc={train_acc:.3f} | val_acc={val_acc:.3f}")


Epoch 1/3 | train_acc=0.722 | val_acc=0.957
Epoch 2/3 | train_acc=0.923 | val_acc=0.902
Epoch 3/3 | train_acc=0.931 | val_acc=0.973


### Test evaluation (baseline vs weighted)
I report accuracy, balanced accuracy, and per-class metrics.


In [18]:
def eval_model(model, loader, name="model"):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            out = model(xb)
            preds = out.argmax(dim=1).cpu().numpy()
            ps.append(preds)
            ys.append(yb.numpy())
    y_true = np.concatenate(ys)
    y_pred = np.concatenate(ps)
    acc = (y_true == y_pred).mean()
    bal = balanced_accuracy_score(y_true, y_pred)
    print(f"{name}: accuracy={acc:.3f}, balanced_acc={bal:.3f}")
    print(classification_report(y_true, y_pred, target_names=train_ds.classes, digits=3))
    return y_true, y_pred

y_true_base, y_pred_base = eval_model(baseline, test_loader, name="baseline")
y_true_w, y_pred_w = eval_model(weighted, test_loader, name="weighted")


baseline: accuracy=0.973, balanced_acc=0.922
                precision    recall  f1-score   support

     Adialer.C      1.000     1.000     1.000        13
     Agent.FYI      1.000     1.000     1.000        13
     Allaple.A      1.000     1.000     1.000       296
     Allaple.L      1.000     1.000     1.000       160
 Alueron.gen!J      1.000     1.000     1.000        21
     Autorun.K      0.000     0.000     0.000        12
       C2LOP.P      0.938     0.938     0.938        16
   C2LOP.gen!g      0.950     0.950     0.950        20
Dialplatform.B      1.000     1.000     1.000        19
     Dontovo.A      1.000     1.000     1.000        17
      Fakerean      1.000     1.000     1.000        39
 Instantaccess      1.000     1.000     1.000        44
    Lolyda.AA1      0.957     1.000     0.978        22
    Lolyda.AA2      1.000     1.000     1.000        19
    Lolyda.AA3      1.000     1.000     1.000        13
     Lolyda.AT      1.000     0.941     0.970        17
  

  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


weighted: accuracy=0.970, balanced_acc=0.918
                precision    recall  f1-score   support

     Adialer.C      1.000     1.000     1.000        13
     Agent.FYI      1.000     1.000     1.000        13
     Allaple.A      0.997     1.000     0.998       296
     Allaple.L      1.000     1.000     1.000       160
 Alueron.gen!J      1.000     1.000     1.000        21
     Autorun.K      0.000     0.000     0.000        12
       C2LOP.P      1.000     0.688     0.815        16
   C2LOP.gen!g      0.944     0.850     0.895        20
Dialplatform.B      1.000     1.000     1.000        19
     Dontovo.A      1.000     1.000     1.000        17
      Fakerean      1.000     0.974     0.987        39
 Instantaccess      1.000     1.000     1.000        44
    Lolyda.AA1      1.000     1.000     1.000        22
    Lolyda.AA2      1.000     1.000     1.000        19
    Lolyda.AA3      1.000     1.000     1.000        13
     Lolyda.AT      1.000     1.000     1.000        17
  

  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


### Sensitivity to superficial artifacts (simple stress test)
I apply a Gaussian blur to test images and measure the performance drop. If accuracy falls sharply, the model may rely on fine-grained visual artifacts.


In [19]:
blur_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.GaussianBlur(kernel_size=5, sigma=1.0),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

test_blur_ds = datasets.ImageFolder(TEST_DIR, transform=blur_tf)
test_blur_loader = DataLoader(test_blur_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

_, _ = eval_model(baseline, test_blur_loader, name="baseline_blur")
_, _ = eval_model(weighted, test_blur_loader, name="weighted_blur")


baseline_blur: accuracy=0.748, balanced_acc=0.799
                precision    recall  f1-score   support

     Adialer.C      1.000     1.000     1.000        13
     Agent.FYI      1.000     1.000     1.000        13
     Allaple.A      0.688     0.939     0.794       296
     Allaple.L      0.000     0.000     0.000       160
 Alueron.gen!J      0.955     1.000     0.977        21
     Autorun.K      0.000     0.000     0.000        12
       C2LOP.P      0.238     0.625     0.345        16
   C2LOP.gen!g      0.528     0.950     0.679        20
Dialplatform.B      1.000     0.947     0.973        19
     Dontovo.A      1.000     1.000     1.000        17
      Fakerean      1.000     0.897     0.946        39
 Instantaccess      1.000     1.000     1.000        44
    Lolyda.AA1      0.846     1.000     0.917        22
    Lolyda.AA2      0.000     0.000     0.000        19
    Lolyda.AA3      1.000     1.000     1.000        13
     Lolyda.AT      1.000     0.941     0.970        

  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


weighted_blur: accuracy=0.748, balanced_acc=0.799
                precision    recall  f1-score   support

     Adialer.C      1.000     1.000     1.000        13
     Agent.FYI      1.000     1.000     1.000        13
     Allaple.A      0.635     0.939     0.757       296
     Allaple.L      0.000     0.000     0.000       160
 Alueron.gen!J      0.955     1.000     0.977        21
     Autorun.K      0.000     0.000     0.000        12
       C2LOP.P      0.706     0.750     0.727        16
   C2LOP.gen!g      0.800     0.800     0.800        20
Dialplatform.B      1.000     0.947     0.973        19
     Dontovo.A      1.000     1.000     1.000        17
      Fakerean      1.000     0.949     0.974        39
 Instantaccess      1.000     1.000     1.000        44
    Lolyda.AA1      0.840     0.955     0.894        22
    Lolyda.AA2      0.000     0.000     0.000        19
    Lolyda.AA3      0.929     1.000     0.963        13
     Lolyda.AT      1.000     0.941     0.970        

  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


### Interpretation (report wording)
If accuracy drops substantially under blur or light perturbations, the CNN is likely using texture/edge artifacts rather than stable malware structure. This suggests that obfuscation or packing could reduce reliability in real deployments.


### Reflection (mandatory)
- **Is malware image classification fragile?** It can be: small visual changes (packing/obfuscation) may alter textures the CNN relies on.
- **Should we trust models we cannot interpret?** Only with caution and validation; opaque features can fail silently under distribution shift.
- **What surprised me / what I would change:** If blur causes large accuracy drops, I would add robustness training or switch to features that are invariant to superficial changes.


### Confusion matrices (save to figures/)
Run this after training to generate plots for the report.


In [20]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Row-normalized (%) confusion matrix with readable annotations

def save_confusion_matrix_pct(model, loader, name):
    y_true, y_pred = [], []
    model.eval()
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            out = model(xb)
            preds = out.argmax(dim=1).cpu().numpy()
            y_pred.append(preds)
            y_true.append(yb.numpy())
    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    cm_pct = cm / cm.sum(axis=1, keepdims=True)

    # Annotate only diagonal to avoid clutter for many classes
    annot = np.full(cm_pct.shape, "", dtype=object)
    for i in range(cm_pct.shape[0]):
        annot[i, i] = f"{cm_pct[i, i]*100:.0f}"

    fig, ax = plt.subplots(figsize=(10, 9))
    sns.heatmap(
        cm_pct,
        cmap="Blues",
        xticklabels=train_ds.classes,
        yticklabels=train_ds.classes,
        annot=annot,
        fmt="",
        cbar=True,
        ax=ax
    )

    # Reduce clutter for many classes by showing every second class label
    if num_classes > 15:
        step = 2
        ax.set_xticks(ax.get_xticks()[::step])
        ax.set_yticks(ax.get_yticks()[::step])
        ax.set_xticklabels([train_ds.classes[i] for i in range(0, num_classes, step)], fontsize=8)
        ax.set_yticklabels([train_ds.classes[i] for i in range(0, num_classes, step)], fontsize=8)
    else:
        ax.tick_params(axis='x', labelsize=8)
        ax.tick_params(axis='y', labelsize=8)

    # Requested label rotations for readability
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)

    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')
    ax.set_title(f"Q3: Confusion matrix (row-normalized, %) - {name}")
    fig.tight_layout()
    fig.savefig(f"../figures/q3_confusion_matrix_{name}_annot.png", dpi=220)
    fig.savefig(f"../figures/q3_confusion_matrix_{name}_blue.png", dpi=220)
    plt.close(fig)

save_confusion_matrix_pct(baseline, test_loader, "baseline")
save_confusion_matrix_pct(weighted, test_loader, "weighted")
